In [ ]:
"""
============================================================
  Lab 02: Prompt Engineering Workshop
  LLM Course — Lecture 2: Prompt Engineering
============================================================

OBJECTIVE:
  Master prompt engineering by learning how to communicate
  effectively with LLMs. You will iterate through 5 prompt
  strategies — from naive to advanced — and see how the SAME
  question produces dramatically different answers.

  NOTE: This lab does NOT use RAG. We provide context manually
  so you can focus 100% on the prompting techniques. In Lab 03,
  you'll combine these skills with automatic retrieval (RAG).

WHAT YOU'LL LEARN:
  - Why naive prompts fail (hallucination, verbosity, wrong format)
  - Zero-shot vs. Few-shot prompting
  - Chain-of-Thought (CoT) reasoning
  - System messages and role definition
  - Structured output (JSON)
  - Prompt anti-patterns to avoid

DURATION: ~35 minutes

PREREQUISITES:
  pip install openai rich python-dotenv
  + Either Ollama running locally OR DeepSeek API key in .env
============================================================
"""

In [ ]:
import os
import sys
import json
from pathlib import Path
from textwrap import dedent

In [ ]:
from dotenv import load_dotenv
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.prompt import Prompt
from rich import box

In [ ]:
console = Console()
load_dotenv(Path(__file__).parent.parent / ".env")

============================================================
LLM CLIENT SETUP (same helper as other labs)
============================================================

In [ ]:
def get_llm_client():
    """Configure the OpenAI-compatible client (DeepSeek or Ollama)."""
    from openai import OpenAI

    # Try DeepSeek first (cloud), then fall back to Ollama (local)
    api_key = os.getenv("DEEPSEEK_API_KEY")

    if api_key and api_key != "your-api-key-here":
        client = OpenAI(
            base_url=os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com"),
            api_key=api_key,
        )
        model = os.getenv("DEEPSEEK_CHAT_MODEL", "deepseek-chat")
        console.print(f"[bold green]✓[/] Using DeepSeek API (model: {model})")
    else:
        client = OpenAI(
            base_url=os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"),
            api_key="ollama",
        )
        model = os.getenv("OLLAMA_CHAT_MODEL", "mistral")
        console.print(f"[bold green]✓[/] Using Ollama (model: {model})")

    return client, model

============================================================
STEP 1: Sample Context (No RAG needed!)
============================================================
EXPLANATION:
  Instead of retrieving context from a vector database,
  we provide it directly. This lets you focus 100% on
  prompt engineering. In Lab 03, you'll learn RAG to
  automate the retrieval part.
============================================================

In [ ]:
SAMPLE_CONTEXTS = {
    "remote_work": """
REMOTE WORK POLICY (Updated January 2024)

Section 1: Eligibility
All full-time employees who have completed their probationary period (90 days) are eligible
for remote work. Part-time employees and contractors are not eligible unless approved by
their department head in writing.

Section 2: Schedule
Eligible employees may work remotely up to 3 days per week. The remaining 2 days must be
spent in the office. Core hours (10:00 AM - 4:00 PM) must be observed regardless of location.
Remote work days must be pre-approved by the direct manager at least 48 hours in advance.

Section 3: Equipment
The company provides a laptop and headset for remote work. Employees are responsible for
their own internet connection (minimum 50 Mbps). A one-time stipend of €500 is available
for home office setup.
""",

    "sick_leave": """
SICK LEAVE POLICY (Updated March 2024)

Section 1: Entitlement
Full-time employees receive 12 paid sick days per calendar year. Unused sick days do NOT
carry over to the next year. Part-time employees receive a pro-rated amount.

Section 2: Notification
Employees must notify their manager by 9:00 AM on the first day of absence. For absences
of 3 or more consecutive days, a medical certificate is required. The certificate must be
submitted to HR within 48 hours of return.

Section 3: Extended Illness
For illnesses exceeding 5 consecutive days, the employee must contact HR directly.
Extended sick leave (beyond 12 days) is handled through the company's disability insurance
program. Employees on extended leave retain their position for up to 6 months.
""",

    "expense": """
TRAVEL & EXPENSE POLICY (Updated February 2024)

Section 1: Air Travel
Economy class is required for all domestic flights. Premium economy is permitted for
international flights exceeding 6 hours. Business class requires VP-level approval.
First class is not reimbursable under any circumstances.

Section 2: Hotels
Maximum hotel rates: €150/night domestic, €250/night international (major cities),
€200/night international (other). Rates above these limits require pre-approval.

Section 3: Meals
Daily meal allowance: €50 domestic, €75 international. Alcohol is not reimbursable.
Receipts are required for all expenses above €25.
""",
}

In [ ]:
def get_context(topic: str = "all") -> str:
    """Get sample context for prompting exercises."""
    if topic in SAMPLE_CONTEXTS:
        return SAMPLE_CONTEXTS[topic]
    return "\n---\n".join(SAMPLE_CONTEXTS.values())

============================================================
STEP 2: The Naive Prompt (Baseline)
============================================================
EXPLANATION:
  This is the simplest possible prompt. No system message,
  no instructions, no format guidance. Just dump everything.
  Watch how the model often gives verbose, unfocused answers
  and sometimes makes things up.
============================================================

In [ ]:
def prompt_naive(client, model: str, query: str, context: str) -> str:
    """Baseline: minimal prompt with no engineering."""
    prompt = f"""Answer the question based on the context.

Context: {context}

Question: {query}

Answer:"""

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=500,
    )
    return response.choices[0].message.content

============================================================
STEP 3: Zero-Shot with System Message
============================================================
EXPLANATION:
  Adding a system message gives the model a "role" and
  clear constraints. This alone dramatically reduces
  hallucination and improves answer quality.
============================================================

In [ ]:
def prompt_zero_shot(client, model: str, query: str, context: str) -> str:
    """Zero-shot: system message with role + constraints."""

    system = dedent("""\
        You are an HR policy assistant for a large company.
        Answer questions ONLY based on the provided context.
        If the context does not contain the answer, say "I don't have enough information to answer this."
        Be concise and direct. Use bullet points for multi-part answers.""")

    user = f"""Context:
{context}

Question: {query}"""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        temperature=0.2,
        max_tokens=400,
    )
    return response.choices[0].message.content

============================================================
STEP 4: Few-Shot Prompting
============================================================
EXPLANATION:
  By providing examples of ideal question-answer pairs,
  we teach the model the exact format and tone we want.
  Notice how every example cites its source and uses bold
  for key values — the model will mimic this pattern.
============================================================

In [ ]:
FEW_SHOT_EXAMPLES = [
    {
        "question": "How many vacation days do new employees get?",
        "answer": "New employees receive **15 days** of paid vacation per year during their first 2 years. After 2 years, this increases to 20 days. (Source: Employee Benefits Policy)"
    },
    {
        "question": "Can I expense a first-class flight?",
        "answer": "No. The company policy requires **economy class** for domestic flights and **premium economy** for international flights over 6 hours. First-class is not reimbursable. (Source: Travel & Expense Policy)"
    },
]

In [ ]:
def prompt_few_shot(client, model: str, query: str, context: str) -> str:
    """Few-shot: provide examples of ideal answers."""

    examples_text = ""
    for ex in FEW_SHOT_EXAMPLES:
        examples_text += f"\nQ: {ex['question']}\nA: {ex['answer']}\n"

    system = dedent("""\
        You are an HR policy assistant. Answer questions based ONLY on the provided context.
        Follow the format shown in the examples: concise, cite the source, use bold for key values.""")

    user = f"""Here are examples of ideal answers:
{examples_text}
---
Context:
{context}

Now answer this question in the same format:
Q: {query}
A:"""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        temperature=0.2,
        max_tokens=400,
    )
    return response.choices[0].message.content

============================================================
STEP 5: Chain-of-Thought (CoT)
============================================================
EXPLANATION:
  CoT forces the model to "think step by step" before
  answering. This dramatically improves accuracy on complex
  questions that require reasoning across multiple sections.

TODO: Implement a CoT prompt!
============================================================

In [ ]:
def prompt_cot(client, model: str, query: str, context: str) -> str:
    """
    Chain-of-Thought: force step-by-step reasoning.

    TODO: Implement this function!
    Steps:
      1. Add a system message instructing step-by-step reasoning
      2. Tell the model to: identify relevant sections, reason, then answer
      3. Require a "Sources:" section at the end

    Hint: Use "Let's think step by step" or
          "First, identify... Then, analyze... Finally, answer..."
    """

    # ──────────────────────────────────────────────
    # TODO: YOUR CODE HERE
    # ──────────────────────────────────────────────

    # Placeholder — replace with your CoT implementation
    return prompt_zero_shot(client, model, query, context)

============================================================
STEP 6: Structured Output (JSON)
============================================================
EXPLANATION:
  For downstream processing (APIs, databases, UIs), we need
  the model to return structured data. This requires very
  explicit format instructions in the prompt.
============================================================

In [ ]:
def prompt_structured(client, model: str, query: str, context: str) -> str:
    """Force the model to return a structured JSON response."""

    system = dedent("""\
        You are an HR policy assistant. Answer questions based ONLY on the provided context.
        You MUST respond in valid JSON format with this exact schema:
        {
            "answer": "your concise answer here",
            "confidence": "high" | "medium" | "low",
            "sources": ["list of source document names used"],
            "relevant_quote": "exact quote from context that supports the answer"
        }
        If you cannot answer, set confidence to "low" and answer to "Information not found." """)

    user = f"""Context:
{context}

Question: {query}

Respond in JSON:"""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        temperature=0.1,
        max_tokens=500,
    )
    return response.choices[0].message.content

============================================================
STEP 7: Prompt Comparison
============================================================

In [ ]:
PROMPT_STRATEGIES = {
    "naive": ("Naive (no engineering)", prompt_naive),
    "zero_shot": ("Zero-Shot + System Message", prompt_zero_shot),
    "few_shot": ("Few-Shot (with examples)", prompt_few_shot),
    "cot": ("Chain-of-Thought", prompt_cot),
    "structured": ("Structured JSON Output", prompt_structured),
}

In [ ]:
TEST_QUERIES = [
    # Simple factual
    "How many days per week can I work remotely?",
    # Requires reading conditions
    "What happens if I'm sick for more than a week?",
    # Requires comparing across documents
    "Compare the remote work and sick leave notification requirements.",
    # Should trigger "I don't know"
    "What is the company's policy on cryptocurrency investments?",
]

In [ ]:
def run_prompt_comparison(client, model: str):
    """Compare all 5 prompt strategies on the same query."""

    query = TEST_QUERIES[0]
    context = get_context("remote_work")

    console.print(f"\n[bold]Test Query:[/] {query}")
    console.print(f"[dim]Context: Remote Work Policy (provided manually)[/]\n")

    for key, (name, func) in PROMPT_STRATEGIES.items():
        console.print(f"{'─'*50}")
        console.print(f"[bold cyan]Strategy: {name}[/]")
        answer = func(client, model, query, context)
        console.print(Panel(answer, border_style="green", title=name))

============================================================
MAIN
============================================================

In [ ]:
def main():
    console.print(Panel(
        "[bold white]Lab 02: Prompt Engineering Workshop[/]\n"
        "[dim]Lecture 2 — Prompt Engineering[/]\n\n"
        "Learn to communicate effectively with LLMs.\n"
        "See how 5 different prompt strategies produce\n"
        "dramatically different answers to the SAME question.",
        title="Hands-on Lab",
        border_style="blue",
    ))

    client, model = get_llm_client()

    # Run the comparison
    run_prompt_comparison(client, model)

    # Interactive mode
    console.print(Panel(
        "Try different strategies on your own questions.\n"
        "Strategies: naive, zero_shot, few_shot, cot, structured\n"
        "Type [cyan]quit[/] to exit.",
        title="Interactive Mode",
        border_style="blue",
    ))

    while True:
        query = Prompt.ask("\n[bold cyan]Your question")
        if query.lower() in ("quit", "exit", "q"):
            break

        strategy = Prompt.ask(
            "[bold]Strategy",
            choices=list(PROMPT_STRATEGIES.keys()),
            default="few_shot",
        )

        context = get_context("all")
        name, func = PROMPT_STRATEGIES[strategy]
        answer = func(client, model, query, context)
        console.print(Panel(answer, border_style="green", title=name))

    console.print(Panel(
        "[bold green]Lab 02 Complete![/]\n\n"
        "Key takeaways:\n"
        "  - Naive prompts lead to hallucination and poor formatting\n"
        "  - System messages define the model's role and constraints\n"
        "  - Few-shot examples teach the model your desired format\n"
        "  - Chain-of-Thought improves reasoning on complex queries\n"
        "  - Structured output enables downstream processing\n\n"
        "[dim]Next: Lab 03 will automate context retrieval with RAG![/]",
        title="Summary",
        border_style="green",
    ))

In [ ]:
if __name__ == "__main__":
    main()